# Find and Re-run Parse-Error Chapters

Scans every subject's `Audit Results` folder, tries to parse each `.json` report as a
JSON array of chunk results, and lists every file that fails — either because the JSON
itself is malformed (e.g. truncated mid-generation) or because it parsed fine but isn't
a list (e.g. an error dict saved from a failed request).

Run this after your main audit pass. It doesn't re-run anything itself — it just tells
you exactly which files are broken so you can delete them and re-run the audit script,
which will pick up only the missing/broken chapters (since it skips files that already
exist and look valid).


In [ ]:
import json
from pathlib import Path
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.width', 300)


## Configuration

Same `BASE_DIR` / `CLASS_SUBJECTS` mapping as your main audit report notebook.


In [2]:
BASE_DIR = Path(r"D:\Ongoing Research Works\AI-Tutor\NCTB-SchoolText")

CLASS_SUBJECTS = {
    "classOne": [
        "processed_chapters_bangla",
        "processed_chapters_english",
        "processed_chapters_math",
    ],
    "classTwo": [
        "processed_chapters_bangla",
        "processed_chapters_english",
        "processed_chapters_math",
    ],
    "classThree": [
        "processed_chapters_bangla",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_english",
        "processed_chapters_hindu_religion",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_science",
    ],
    "classFour": [
        "processed_chapters_bangla",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_english",
        "processed_chapters_hindu_religion",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_science",
    ],
    "classFive": [
        "processed_chapters_bangla",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_eng",
        "processed_chapters_hindu_religion",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_science",
    ],
    "classSix": [
        "processed_chapters_agriculture",
        "processed_chapters_arabic",
        "processed_chapters_arts_and_crafts",
        "processed_chapters_bangla",
        "processed_chapters_bangla_grammar",
        "processed_chapters_bangla_rapidreader",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_english",
        "processed_chapters_english_grammar",
        "processed_chapters_hindu_religion",
        "processed_chapters_home_science",
        "processed_chapters_ict",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_music",
        "processed_chapters_pali",
        "processed_chapters_physical_education",
        "processed_chapters_sanskrit",
        "processed_chapters_science",
        "processed_chapters_work_and_life",
    ],
    "classSeven": [
        "processed_chapters_agriculture",
        "processed_chapters_arabic",
        "processed_chapters_arts_and_crafts",
        "processed_chapters_bangla",
        "processed_chapters_bangla_grammar",
        "processed_chapters_bangla_rapidreader",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_english",
        "processed_chapters_english_grammar",
        "processed_chapters_hindu_religion",
        "processed_chapters_home_science",
        "processed_chapters_ict",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_music",
        "processed_chapters_pali",
        "processed_chapters_physical_education",
        "processed_chapters_sanskrit",
        "processed_chapters_science",
        "processed_chapters_work_and_life",
    ],
    "classEight": [
        "processed_chapters_agriculture",
        "processed_chapters_arabic",
        "processed_chapters_arts_and_crafts",
        "processed_chapters_bangla",
        "processed_chapters_bangla_grammar",
        "processed_chapters_bangla_rapidreader",
        "processed_chapters_bgs",
        "processed_chapters_buddhist_religion",
        "processed_chapters_christian_religion",
        "processed_chapters_english",
        "processed_chapters_english_grammar",
        "processed_chapters_hindu_religion",
        "processed_chapters_home_science",
        "processed_chapters_ict",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_music",
        "processed_chapters_pali",
        "processed_chapters_physical_education",
        "processed_chapters_sanskrit",
        "processed_chapters_science",
        "processed_chapters_work_and_life",
    ],
    "classNineTen": [
        "processed_chapters_accounting",
        "processed_chapters_agriculture",
        "processed_chapters_arabic",
        "processed_chapters_arts_and_crafts",
        "processed_chapters_bangla",
        "processed_chapters_bangla_grammar",
        "processed_chapters_bangla_rapidreader",
        "processed_chapters_bgs",
        "processed_chapters_biology_secondary",
        "processed_chapters_buddhist_religion",
        "processed_chapters_business_entrepreneurship",
        "processed_chapters_career_education",
        "processed_chapters_chemistry_secondary",
        "processed_chapters_christian_religion",
        "processed_chapters_civics",
        "processed_chapters_economics",
        "processed_chapters_english",
        "processed_chapters_english_grammar",
        "processed_chapters_finance",
        "processed_chapters_geography",
        "processed_chapters_higher_math",
        "processed_chapters_hindu_religion",
        "processed_chapters_history",
        "processed_chapters_home_science",
        "processed_chapters_ict",
        "processed_chapters_islam",
        "processed_chapters_math",
        "processed_chapters_music",
        "processed_chapters_pali",
        "processed_chapters_physical_education",
        "processed_chapters_physics_secondary",
        "processed_chapters_sanskrit",
        "processed_chapters_science",
    ],
}


## Scan function

For every `.json` file in every subject's `Audit Results` folder:
- If it fails to parse as JSON at all → flagged as `Invalid JSON` (e.g. truncated mid-generation).
- If it parses but isn't a list → flagged as `Not a list` (e.g. an error dict saved from a
  failed HTTP request or empty-response case).
- Also checks whether the original source `.jsonl` chapter file still exists, so you know
  re-running is actually possible before you delete anything.


In [3]:
def get_subject_name(subject_folder: str) -> str:
    return subject_folder.replace("processed_chapters_", "", 1)


def scan_subject_for_parse_errors(class_name, subject_folder, subject_dir):
    subject_name = get_subject_name(subject_folder)
    audit_dir = subject_dir / "Audit Results"
    failures = []

    if not audit_dir.exists():
        return failures

    for audit_file in sorted(audit_dir.glob("*.json")):
        reason = None
        try:
            content = json.loads(audit_file.read_text(encoding="utf-8"))
            if not isinstance(content, list):
                reason = f"Not a list (parsed as {type(content).__name__} — likely an error response)"
        except json.JSONDecodeError as e:
            reason = f"Invalid JSON: {e}"

        if reason:
            source_jsonl = subject_dir / (audit_file.stem + ".jsonl")
            failures.append({
                "class": class_name,
                "subject": subject_name,
                "audit_file": str(audit_file),
                "source_jsonl": str(source_jsonl),
                "source_jsonl_exists": source_jsonl.exists(),
                "reason": reason,
            })

    return failures


## Run the scan across all classes and subjects


In [4]:
all_failures = []
for class_name, subject_folders in CLASS_SUBJECTS.items():
    for subject_folder in subject_folders:
        subject_dir = BASE_DIR / class_name / subject_folder
        if not subject_dir.exists():
            continue
        all_failures.extend(scan_subject_for_parse_errors(class_name, subject_folder, subject_dir))

failures_df = pd.DataFrame(all_failures)
print(f"Total parse-error files found: {len(failures_df)}")
failures_df


Total parse-error files found: 89


,class,subject,audit_file,source_jsonl,source_jsonl_exists,reason
0,classOne,bangla,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,True,Invalid JSON: Invalid \escape: line 8 column 2...
1,classOne,bangla,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,True,Invalid JSON: Expecting ':' delimiter: line 17...
2,classTwo,english,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,True,Invalid JSON: Invalid \escape: line 62 column ...
3,classTwo,english,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,True,Invalid JSON: Invalid \escape: line 44 column ...
4,classThree,math,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,True,Invalid JSON: Invalid \escape: line 80 column ...
5,classThree,science,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,True,Invalid JSON: Invalid \escape: line 53 column ...
6,classThree,science,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,True,Invalid JSON: Invalid \escape: line 134 column...
7,classFour,bangla,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,True,Invalid JSON: Invalid control character at: li...
8,classFour,bangla,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,True,"Invalid JSON: Expecting ',' delimiter: line 11..."
9,classFour,bgs,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,D:\Ongoing Research Works\AI-Tutor\NCTB-School...,True,Invalid JSON: Unterminated string starting at:...


## Summary by class / subject

Quick view of where the failures are concentrated — useful for spotting whether a
particular subject or class needs a closer look (e.g. consistently longer chapters
hitting the token limit).


In [5]:
if not failures_df.empty:
    summary = failures_df.groupby(["class", "subject"], as_index=False).size()
    summary = summary.rename(columns={"size": "parse_error_count"})
    print(summary.sort_values("parse_error_count", ascending=False).to_string(index=False))
else:
    print("No parse errors found — everything parsed cleanly.")


       class             subject  parse_error_count
   classFour             science                  5
classNineTen             finance                  4
classNineTen                math                  3
   classFive  christian_religion                  3
   classFour              bangla                  2
   classFour   buddhist_religion                  2
   classFive                 bgs                  2
classNineTen         agriculture                  2
classNineTen chemistry_secondary                  2
classNineTen           geography                  2
  classEight     english_grammar                  2
  classEight      hindu_religion                  2
classNineTen         higher_math                  2
classNineTen             history                  2
    classOne              bangla                  2
    classSix                pali                  2
    classSix             science                  2
  classThree             science                  2
    classTwo

## Check: any failures whose source `.jsonl` is missing?

These can't simply be re-run — the original chapter file itself is gone, which is a
different problem worth investigating separately.


In [6]:
if not failures_df.empty:
    missing_source = failures_df[~failures_df["source_jsonl_exists"]]
    if not missing_source.empty:
        print(f"WARNING: {len(missing_source)} failed audit file(s) have no matching source .jsonl:")
        print(missing_source[["class", "subject", "source_jsonl"]].to_string(index=False))
    else:
        print("All failed audit files have a matching source .jsonl — safe to re-run.")


All failed audit files have a matching source .jsonl — safe to re-run.


## Delete the broken audit files so a re-run picks them up

Your main audit script skips any `.jsonl` whose corresponding `.json` output already
exists. Deleting only the broken output files (not the source `.jsonl`) means re-running
the audit script will naturally reprocess just these chapters and leave everything else
alone.

**This actually deletes files.** Set `CONFIRM_DELETE = True` below only when you're ready.


In [ ]:
CONFIRM_DELETE = False  # flip to True to actually delete the broken files listed above

if CONFIRM_DELETE and not failures_df.empty:
    deleted = 0
    for path_str in failures_df["audit_file"]:
        path = Path(path_str)
        if path.exists():
            path.unlink()
            deleted += 1
    print(f"Deleted {deleted} broken audit file(s). Re-run your audit script now.")
elif not CONFIRM_DELETE:
    print("CONFIRM_DELETE is False — nothing deleted. Review the list above first.")
else:
    print("No parse-error files to delete.")
